<a href="https://colab.research.google.com/github/jhuangjennifer/573ChineseEnglishSummarization/blob/jjnhuang-work/notebooks/models/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import BartTokenizer, BartModel
import torch

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')

model = BartModel.from_pretrained(
    'facebook/bart-large',
    force_download=True
)

inputs = tokenizer("Hello, my dog is cute", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

last_hidden_states = outputs.last_hidden_state
print("model shape:", last_hidden_states.shape)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

print("library load")

In [ ]:
pip install transformers

In [ ]:
# dataset check
DATASET_NAME = "XSAMSum"   # you can change this "XMediaSum40k"
MODEL_NAME = "facebook/bart-large"
CACHE_DIR = "./hf_cache"
BASE_DIR = f"sample_data"

In [ ]:
# dataset load
import os
from datasets import load_dataset

#BASE_DIR = f"../../data/raw/{DATASET_NAME}"

data_files = {
    "train": os.path.join(BASE_DIR, "train.json"),
    "validation": os.path.join(BASE_DIR, "val.json"),
    "test": os.path.join(BASE_DIR, "test.json"),
}

print(data_files)

dataset_dict = load_dataset("json", data_files=data_files)
dataset_dict

In [ ]:
import pandas as pd

# DataFrame
df_train = dataset_dict['train'].to_pandas()

# result
display(df_train.head())

In [ ]:
import pandas as pd

# convert into DataFrame
df_train = dataset_dict['train'].to_pandas()
df_val = dataset_dict['validation'].to_pandas()
df_test = dataset_dict['test'].to_pandas()

# result check
print(f"Train rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Test rows: {len(df_test)}")

In [ ]:
display(df_train.head())
display(df_val.head())
display(df_test.head())

In [ ]:
dfs = {split: dataset.to_pandas() for split, dataset in dataset_dict.items()}

display(dfs['train'].head())

In [ ]:
# pick random sample
sample = df_train.sample(1)
print(f"dialogue: {sample['dialogue'].values[0]}")
print(f"summary: {sample['summary'].values[0]}")
print(f"summary_de: {sample['summary_de'].values[0]}")
print(f"summary_z: {sample['summary_zh'].values[0]}")

In [ ]:
import pandas as pd
import re

# 1. Text cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return text
    text = text.replace('\r', '')          # Remove carriage returns
    text = re.sub(r'\s+', ' ', text)       # Replace multiple spaces with a single space
    return text.strip()                    # Remove leading and trailing spaces

# 2. Integrated preprocessing pipeline function
def preprocess_to_clean_df(dataset_split):
    """
    Takes a Hugging Face Dataset as input, extracts only the 'dialogue' and 'summary' columns,
    and returns a DataFrame with missing values and duplicates removed, and text normalized.
    """
    # Convert to Pandas DataFrame
    df = dataset_split.to_pandas()

    # Select only the necessary columns
    df = df[['dialogue', 'summary', 'summary_zh']].copy()

    # Drop missing values
    df = df.dropna(subset=['dialogue', 'summary', 'summary_zh'])

    # Apply text normalization
    df['dialogue'] = df['dialogue'].apply(clean_text)
    df['summary'] = df['summary'].apply(clean_text)
    df['summary_zh'] = df['summary_zh'].apply(clean_text)

    # Remove duplicate data and reset index
    df = df.drop_duplicates().reset_index(drop=True)

    return df

# 3. Create DataFrames with updated variable names
train_data = preprocess_to_clean_df(dataset_dict['train'])
val_data = preprocess_to_clean_df(dataset_dict['validation'])
test_data = preprocess_to_clean_df(dataset_dict['test'])

# 4. Check results
print("Preprocessing results!")
print(f"train_data: {train_data.shape}")
print(f"val_data: {val_data.shape}")
print(f"test_data: {test_data.shape}\n")

# Check a sample
display(train_data.head())

In [ ]:
train_data['dialogue'].loc[0]

In [ ]:
train_data[['dialogue', 'summary', 'summary_zh']].head()

In [ ]:
test_data[['dialogue', 'summary']].head()

In [ ]:
val_data[['dialogue', 'summary']].head()

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from datasets import Dataset, DatasetDict
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# 1. Convert Pandas DataFrames to Hugging Face Dataset format
# Utilizing the previously created train_data, val_data, and test_data.
hg_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_data),
    "validation": Dataset.from_pandas(val_data),
    "test": Dataset.from_pandas(test_data)
})

# 2. Load the Model and Tokenizer
model_name = "facebook/bart-large"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 3. Tokenization preprocessing function (applying max_length based on the paper)
def tokenize_function_translated(examples):
    model_inputs = tokenizer(
        examples["dialogue"],
        max_length=1024,
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary_zh"],
        max_length=150,
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Map tokenization to the entire dataset (batched=True for speed optimization)
print("start tokenizing")
tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)
translated_tokenized_datasets = hg_dataset.map(tokenize_function_translated, batched=True)
print("tokenization complete!")

# 4. Set up Data Collator (Dynamic padding within batches)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. M4 Pro customized training setup (Implementing the paper's batch size of 24)
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-dialogue-summary-final",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,       # Reduced from 8 to 2
    gradient_accumulation_steps=12,      # Increased to 12 (2 * 12 = 24, same as paper)
    per_device_eval_batch_size=2,        # Reduced for evaluation as well
    gradient_checkpointing=True,         # Crucial: Trades compute for memory
    num_train_epochs=20,
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=False,
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"], # Add validation set!
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Ready to do trainer.train()")

In [ ]:
# =====================================================================
# STEP 1: SELECT SMALL SUBSET FOR TESTING
# =====================================================================
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["validation"]

# =====================================================================
# STEP 2: MODIFIED TRAINING ARGUMENTS FOR SPEED
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-test-run",
    eval_strategy="no",
    learning_rate=2e-5,

    # Increase batch size slightly to speed up M4 Pro (since we have 48GB)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=6,       # 4 * 6 = 24 (Still effective batch size 24)

    # Use max_steps for a quick exit
    max_steps=50,                        # Stop training exactly at 50 steps
    num_train_epochs=1,                  # Ensure it doesn't run more than 1 epoch

    optim="adafactor",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=10,                    # Log every 10 steps to see progress
    save_strategy="no",                  # Don't waste time saving checkpoints during test
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    report_to="none"
)

# =====================================================================
# STEP 3: INITIALIZE & RUN
# =====================================================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # Use the sliced small dataset
    processing_class=tokenizer,
    data_collator=data_collator,
)

print(" Starting a high-speed test run (50 steps)...")
trainer.train()

In [ ]:
pip install evaluate

In [ ]:
pip install rouge_score

In [ ]:
# Load ROUGE metric
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    print("Inside compute metrics")
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    # Decode predictions and labels into text
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE scores
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# Run evaluation on the test set
print(" Evaluating on the test set...")
results = trainer.evaluate(eval_dataset=translated_tokenized_datasets["test"], metric_key_prefix="test", compute_metrics=compute_metrics)
print(results)

In [ ]:
# Initialize translation tokenizer and model: https://huggingface.co/Helsinki-NLP/opus-mt-en-zhhttps://huggingface.co/Helsinki-NLP/opus-mt-en-zh

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

translate_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-zh")
translate_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-zh")

In [ ]:
def summarize_dialogue(text):
    # 1. Prepare input - this returns a dict containing 'input_ids' and 'attention_mask'
    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True).to(model.device)

    # 2. Generate summary - explicitly pass both input_ids and attention_mask
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"], # Crucial for MPS compatibility
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

    # 3. Decode summary
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # 4. Translate summary
    tokenized_summary = translate_tokenizer(summary, return_tensors="pt").to(translate_model.device)
    output = translate_model.generate(**tokenized_summary)

    # 5. Decode and return translated summary
    decoded_output = translate_tokenizer.decode(output[0], skip_special_tokens=True)
    return decoded_output

# Test with the sample dialogue again
sample_text = "Yun: Hey, did you finish the training? Yunmin: Yes, it's done! Yun: Great, what's next?"
print(f"Summary: {summarize_dialogue(sample_text)}")

In [ ]:
# Evaluation

# Install dependencies
!pip install pyonmttok jieba six nltk absl-py

# Install the xl-sum multilingual rouge_score fork
# This overwrites the standard rouge_score package with the multilingual version.
!pip install git+https://github.com/csebuetnlp/xl-sum.git#subdirectory=multilingual_rouge_scoring

# BERTScore
!pip install bert-score transformers

In [ ]:
# Configuration

# ClidSum paper uses chinese-bert-wwm-ext for Chinese BERTScore.
# HuggingFace model ID:
BERTSCORE_MODEL = "hfl/chinese-bert-wwm-ext"

# ClidSum paper uses R1/R2/R-L
ROUGE_TYPES = ["rouge1", "rouge2", "rougeL"]

In [ ]:
references = dataset_dict['test']['summary_zh']
predictions = [summarize_dialogue(text) for text in dataset_dict['test']['dialogue']]

# Sanity checks
assert len(predictions) == len(references)

summary_eval_pair_scores = {
    'references':  references,
    'predictions': predictions,
    'rougescore':  [],   # one dict per pair: {'rouge1': float, 'rouge2': float, 'rougeL': float}
    'bertscore':   [],   # one F1 float per pair
}


In [ ]:
# ROUGE Evaluation for Chinese
from rouge_score import rouge_scorer

def compute_rouge(
    predictions: list[str],
    references: list[str],
    rouge_types: list[str],
    language: str = "zh",
) -> tuple[dict, list[dict]]:

    scorer = rouge_scorer.RougeScorer(
        rouge_types=rouge_types,
        lang=language,
    )

    pair_scores = []
    aggregated = {rt: 0.0 for rt in rouge_types}

    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)  # (reference, hypothesis)
        pair = {rt: round(scores[rt].fmeasure * 100, 2) for rt in rouge_types}
        pair_scores.append(pair)
        for rt in rouge_types:
            aggregated[rt] += scores[rt].fmeasure  # extract only F1 score

    n = len(predictions)
    corpus_scores = {rt: round(aggregated[rt] / n * 100, 2) for rt in rouge_types}

    return corpus_scores, pair_scores


rouge_corpus_scores, rouge_pair_scores = compute_rouge(predictions, references, ROUGE_TYPES, language="zh")

summary_eval_pair_scores['rougescore'] = rouge_pair_scores

print(f"  ROUGE-1 (R1) : {rouge_corpus_scores['rouge1']:.2f}")
print(f"  ROUGE-2 (R2) : {rouge_corpus_scores['rouge2']:.2f}")
print(f"  ROUGE-L (R-L): {rouge_corpus_scores['rougeL']:.2f}")


In [ ]:
# BERTScore Evaluation for Chinese
from bert_score import BERTScorer

def compute_bertscore(
    predictions: list[str],
    references: list[str],
    model_type: str,
    lang: str = "zh",
    batch_size: int = 32,
    verbose: bool = True,
) -> tuple[dict, list]:

    scorer = BERTScorer(
        model_type=model_type,
        lang=lang,
        batch_size=batch_size,
        num_layers=8,  # matches bert-base-chinese optimal layer per BERTScore paper
    )

    # Fix the OverflowError
    scorer._tokenizer.model_max_length = 512

    P, R, F1 = scorer.score(
        predictions,
        list(references),
        verbose=verbose,
        batch_size=batch_size,
    )

    pair_scores = [round(F1[i].item() * 100, 2) for i in range(len(predictions))]
    corpus_scores = {"f1": round(F1.mean().item() * 100, 2)}

    return corpus_scores, pair_scores

bert_corpus_scores, bert_pair_scores = compute_bertscore(
    predictions,
    references,
    model_type=BERTSCORE_MODEL,
    lang="zh",
    batch_size=32,
)

summary_eval_pair_scores['bertscore'] = bert_pair_scores

print(f"  F1 (B-S)  : {bert_corpus_scores['f1']}")

In [ ]:
# Combine and Save Results

import pandas as pd

# combine corpus level scores
corpus_df = pd.DataFrame(rouge_corpus_scores | bert_corpus_scores, index=[0])
print(f'Corpus Eval Scores of {DATASET_NAME}')
display(corpus_df)

# combine individual pair scores
rouge_df = pd.DataFrame(summary_eval_pair_scores['rougescore'])  # cols: rouge1, rouge2, rougeL
bert_df  = pd.DataFrame({'bs_f1': summary_eval_pair_scores['bertscore']})

pair_results_df = pd.DataFrame({
    'reference':  summary_eval_pair_scores['references'],
    'prediction': summary_eval_pair_scores['predictions'],
} | rouge_df.to_dict('list') | bert_df.to_dict('list'))

# display analysis
print("\n── BERTScore F1 distribution ──")
print(pair_results_df['bs_f1'].describe().round(2))

TOP_N_WORST = 5
worst = pair_results_df.nsmallest(TOP_N_WORST, 'rougeL')
print(f'\n── Top {TOP_N_WORST} worst examples by ROUGE-L of {DATASET_NAME} ──')
display(worst)

# save results
corpus_df.to_csv(f"corpus_scores_{DATASET_NAME}.csv", index=False, encoding="utf-8-sig")
pair_results_df.to_csv(f"pair_scores_{DATASET_NAME}.csv", index=True, encoding="utf-8-sig")